# Vend + Consumption Overlap EDA v2

This notebook tries all four practical overlap keys across the processed Vend and Consumption parquet files:

- `consumernumber`
- `consumernumber_normalized`
- `meterno`
- `meterno_normalized`

What it does:

- profiles overlap for all four keys
- shows which key has the strongest actual intersection
- lets you override the selected key if you want
- builds or loads a saved overlap parquet for the chosen key
- runs EDA and daily charts on that overlap cohort

For the raw keys (`consumernumber`, `meterno`), the notebook only trims whitespace and drops blank or null-like text. It does not apply the scientific-notation-safe normalization used by the processed normalized columns.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and config/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CANDIDATE_KEYS = [
    "consumernumber",
    "consumernumber_normalized",
    "meterno",
    "meterno_normalized",
]
DEFAULT_PREFERENCE_ORDER = [
    "consumernumber_normalized",
    "consumernumber",
    "meterno_normalized",
    "meterno",
]

vend_df = pd.read_parquet(PROCESSED_DIR / "vend.parquet")
consumption_df = pd.read_parquet(PROCESSED_DIR / "consumption.parquet")
master_df = pd.read_parquet(PROCESSED_DIR / "consumer_master.parquet")

NULL_LIKE_TEXT = {"", "na", "n/a", "nan", "null", "none", "<na>"}


def available_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]


def prepare_key_series(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series(pd.NA, index=df.index, dtype="string")

    series = df[column].astype("string")
    text = series.fillna("").str.strip()

    # For raw keys we only trim and remove null-like text.
    # For normalized keys we preserve the processed values as they already stand.
    text = text.where(~text.str.lower().isin(NULL_LIKE_TEXT), pd.NA)
    return text.astype("string")


def prepared_key_set(df: pd.DataFrame, column: str) -> set[str]:
    return set(prepare_key_series(df, column).dropna().tolist())


def overlap_summary(left_df: pd.DataFrame, right_df: pd.DataFrame, column: str, left_name: str, right_name: str) -> dict:
    left_series = prepare_key_series(left_df, column)
    right_series = prepare_key_series(right_df, column)
    left_keys = set(left_series.dropna().tolist())
    right_keys = set(right_series.dropna().tolist())
    overlap_keys = left_keys & right_keys

    left_rows_in_overlap = int(left_series.isin(overlap_keys).sum()) if overlap_keys else 0
    right_rows_in_overlap = int(right_series.isin(overlap_keys).sum()) if overlap_keys else 0

    return {
        "key_column": column,
        "left_dataset": left_name,
        "right_dataset": right_name,
        "left_distinct_non_null_keys": len(left_keys),
        "right_distinct_non_null_keys": len(right_keys),
        "overlap_distinct_keys": len(overlap_keys),
        "left_rows_in_overlap": left_rows_in_overlap,
        "right_rows_in_overlap": right_rows_in_overlap,
        "overlap_pct_of_left_distinct": round(len(overlap_keys) / len(left_keys) * 100, 2) if left_keys else 0.0,
        "overlap_pct_of_right_distinct": round(len(overlap_keys) / len(right_keys) * 100, 2) if right_keys else 0.0,
        "sample_overlap_values": " | ".join(list(sorted(overlap_keys))[:10]),
    }


diagnostic_df = pd.DataFrame([
    overlap_summary(vend_df, consumption_df, key, "vend", "consumption")
    for key in CANDIDATE_KEYS
])

display(diagnostic_df.sort_values(["overlap_distinct_keys", "left_rows_in_overlap", "right_rows_in_overlap"], ascending=False).reset_index(drop=True))

## Key selection

Set `MANUAL_KEY_COLUMN` to one of the candidate keys if you want to force a specific option. Leave it as `None` to let the notebook choose the strongest overlap automatically.

In [ ]:
MANUAL_KEY_COLUMN = None

priority_rank = {key: idx for idx, key in enumerate(DEFAULT_PREFERENCE_ORDER)}
diagnostic_ranked = diagnostic_df.copy()
diagnostic_ranked["priority_rank"] = diagnostic_ranked["key_column"].map(priority_rank)
diagnostic_ranked = diagnostic_ranked.sort_values(
    ["overlap_distinct_keys", "left_rows_in_overlap", "right_rows_in_overlap", "priority_rank"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

if MANUAL_KEY_COLUMN is not None:
    SELECTED_KEY_COLUMN = MANUAL_KEY_COLUMN
    SELECTED_KEY_REASON = f"Using manual override: {MANUAL_KEY_COLUMN}."
else:
    SELECTED_KEY_COLUMN = diagnostic_ranked.loc[0, "key_column"]
    SELECTED_KEY_REASON = "Using the key with the strongest overlap, with the configured preference order used only as a tiebreaker."

SELECTED_KEY_SUFFIX = SELECTED_KEY_COLUMN.replace("_normalized", "_norm")
OVERLAP_ACTIVITY_PATH = PROCESSED_DIR / f"vend_consumption_overlap_activity_{SELECTED_KEY_SUFFIX}.parquet"

PROFILE_COLUMNS = [
    "selected_key_value",
    "selected_key_column",
    "consumer_master_id",
    "master_consumername",
    "master_consumernumber",
    "master_meterno",
    "master_tariffcode",
    "master_area_type",
    "master_connection_type",
    "master_accountingmode",
    "master_feedercode",
    "master_dtcode",
    "master_substationname",
    "master_meterbalance",
    "master_profile_status",
]
ACTIVITY_COLUMNS = PROFILE_COLUMNS + [
    "activity_source",
    "activity_date",
    "source_consumernumber",
    "source_consumernumber_normalized",
    "source_meterno",
    "source_meterno_normalized",
    "vend_amount_rs",
    "kwh",
    "kvah",
]

display(diagnostic_ranked)
print(f"Selected key: {SELECTED_KEY_COLUMN}")
print(SELECTED_KEY_REASON)
print(f"Overlap parquet path: {OVERLAP_ACTIVITY_PATH}")

if int(diagnostic_ranked.loc[0, "overlap_distinct_keys"]) == 0:
    print("Warning: all four candidate keys show zero overlap, so the saved overlap file will remain empty until the source identifiers can be reconciled.")

## Sample overlap rows for the selected key

This gives a quick visual check of how the chosen key looks in both source datasets.

In [ ]:
vend_selected_key = prepare_key_series(vend_df, SELECTED_KEY_COLUMN)
consumption_selected_key = prepare_key_series(consumption_df, SELECTED_KEY_COLUMN)
selected_overlap_keys = set(vend_selected_key.dropna().tolist()) & set(consumption_selected_key.dropna().tolist())

vend_selected_sample = vend_df[vend_selected_key.isin(selected_overlap_keys)][[
    column for column in [
        "consumernumber",
        "consumernumber_normalized",
        "meterno",
        "meterno_normalized",
        "transactionamount",
        "vend_date",
    ]
    if column in vend_df.columns
]].head(10)

consumption_selected_sample = consumption_df[consumption_selected_key.isin(selected_overlap_keys)][[
    column for column in [
        "consumernumber",
        "consumernumber_normalized",
        "meterno",
        "meterno_normalized",
        "kwh_consumption",
        "date",
    ]
    if column in consumption_df.columns
]].head(10)

print(f"Distinct overlap keys for selected key: {len(selected_overlap_keys):,}")
display(vend_selected_sample)
display(consumption_selected_sample)

In [ ]:
def build_master_lookup(master_df: pd.DataFrame, key_column: str) -> pd.DataFrame:
    master_key = prepare_key_series(master_df, key_column)
    master_work = master_df.copy()
    master_work["selected_key_value"] = master_key
    master_work = master_work[master_work["selected_key_value"].notna()].copy()

    if master_work.empty:
        return pd.DataFrame(columns=PROFILE_COLUMNS)

    key_counts = master_work["selected_key_value"].value_counts(dropna=False)
    unique_keys = set(key_counts[key_counts == 1].index.tolist())
    duplicate_keys = set(key_counts[key_counts > 1].index.tolist())

    unique_lookup = master_work[master_work["selected_key_value"].isin(unique_keys)].copy()
    unique_lookup = unique_lookup.rename(columns={
        "consumername": "master_consumername",
        "consumernumber": "master_consumernumber",
        "meterno": "master_meterno",
        "tariffcode": "master_tariffcode",
        "area_type": "master_area_type",
        "connection_type": "master_connection_type",
        "accountingmode": "master_accountingmode",
        "feedercode": "master_feedercode",
        "dtcode": "master_dtcode",
        "substationname": "master_substationname",
        "meterbalance": "master_meterbalance",
    })
    unique_lookup["selected_key_column"] = key_column
    unique_lookup["master_profile_status"] = "unique_master_match"

    keep = [column for column in PROFILE_COLUMNS if column in unique_lookup.columns]
    unique_lookup = unique_lookup[keep].drop_duplicates(subset=["selected_key_value"]).copy()

    duplicate_lookup = pd.DataFrame({"selected_key_value": list(duplicate_keys)})
    if not duplicate_lookup.empty:
        duplicate_lookup["selected_key_column"] = key_column
        duplicate_lookup["master_profile_status"] = "duplicate_master_match"

    lookup = pd.concat([unique_lookup, duplicate_lookup], ignore_index=True, sort=False)
    for column in PROFILE_COLUMNS:
        if column not in lookup.columns:
            lookup[column] = pd.NA
    return lookup[PROFILE_COLUMNS].copy()


def build_or_load_overlap_activity(output_path: Path, force_rebuild: bool = False) -> tuple[pd.DataFrame, str, dict]:
    if output_path.exists() and not force_rebuild:
        activity_df = pd.read_parquet(output_path)
        missing_columns = [column for column in ACTIVITY_COLUMNS if column not in activity_df.columns]
        if not missing_columns:
            return activity_df, "loaded_existing", {"note": "Loaded existing overlap parquet."}

    vend_key = prepare_key_series(vend_df, SELECTED_KEY_COLUMN)
    consumption_key = prepare_key_series(consumption_df, SELECTED_KEY_COLUMN)
    overlap_keys = set(vend_key.dropna().tolist()) & set(consumption_key.dropna().tolist())
    master_lookup = build_master_lookup(master_df, SELECTED_KEY_COLUMN)

    vend_activity = vend_df.copy()
    vend_activity["selected_key_value"] = vend_key
    vend_activity = vend_activity[vend_activity["selected_key_value"].isin(overlap_keys)].copy()
    vend_activity["selected_key_column"] = SELECTED_KEY_COLUMN
    vend_activity["activity_source"] = "vend"
    vend_activity["activity_date"] = pd.to_datetime(vend_activity.get("vend_date"), errors="coerce")
    vend_activity["source_consumernumber"] = vend_activity.get("consumernumber")
    vend_activity["source_consumernumber_normalized"] = vend_activity.get("consumernumber_normalized")
    vend_activity["source_meterno"] = vend_activity.get("meterno")
    vend_activity["source_meterno_normalized"] = vend_activity.get("meterno_normalized")
    vend_activity["vend_amount_rs"] = pd.to_numeric(vend_activity.get("transactionamount"), errors="coerce")
    vend_activity["kwh"] = pd.NA
    vend_activity["kvah"] = pd.NA
    vend_activity = vend_activity.drop(columns=[column for column in PROFILE_COLUMNS if column in vend_activity.columns and column not in {"selected_key_value", "selected_key_column"}], errors="ignore")
    vend_activity = vend_activity.merge(master_lookup, how="left", on=["selected_key_value", "selected_key_column"])
    vend_activity["master_profile_status"] = vend_activity["master_profile_status"].fillna("no_master_profile")

    consumption_activity = consumption_df.copy()
    consumption_activity["selected_key_value"] = consumption_key
    consumption_activity = consumption_activity[consumption_activity["selected_key_value"].isin(overlap_keys)].copy()
    consumption_activity["selected_key_column"] = SELECTED_KEY_COLUMN
    consumption_activity["activity_source"] = "consumption"
    consumption_activity["activity_date"] = pd.to_datetime(consumption_activity.get("date"), errors="coerce")
    consumption_activity["source_consumernumber"] = consumption_activity.get("consumernumber")
    consumption_activity["source_consumernumber_normalized"] = consumption_activity.get("consumernumber_normalized")
    consumption_activity["source_meterno"] = consumption_activity.get("meterno")
    consumption_activity["source_meterno_normalized"] = consumption_activity.get("meterno_normalized")
    consumption_activity["vend_amount_rs"] = pd.NA
    consumption_activity["kwh"] = pd.to_numeric(consumption_activity.get("kwh_consumption"), errors="coerce")
    consumption_activity["kvah"] = pd.to_numeric(consumption_activity.get("kvah_consumption"), errors="coerce")
    consumption_activity = consumption_activity.drop(columns=[column for column in PROFILE_COLUMNS if column in consumption_activity.columns and column not in {"selected_key_value", "selected_key_column"}], errors="ignore")
    consumption_activity = consumption_activity.merge(master_lookup, how="left", on=["selected_key_value", "selected_key_column"])
    consumption_activity["master_profile_status"] = consumption_activity["master_profile_status"].fillna("no_master_profile")

    vend_activity = vend_activity[[column for column in ACTIVITY_COLUMNS if column in vend_activity.columns]].copy()
    consumption_activity = consumption_activity[[column for column in ACTIVITY_COLUMNS if column in consumption_activity.columns]].copy()

    activity_df = pd.concat([vend_activity, consumption_activity], ignore_index=True)
    for column in ACTIVITY_COLUMNS:
        if column not in activity_df.columns:
            activity_df[column] = pd.NA
    activity_df = activity_df[ACTIVITY_COLUMNS].copy()
    activity_df = activity_df.sort_values(["activity_date", "selected_key_value", "activity_source"], kind="stable")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    activity_df.to_parquet(output_path, index=False)

    meta = {
        "note": f"Built overlap parquet using {SELECTED_KEY_COLUMN}.",
        "matched_keys": int(len(overlap_keys)),
        "selected_key": SELECTED_KEY_COLUMN,
    }
    return activity_df, "built_new", meta

In [ ]:
FORCE_REBUILD = False

activity_df, load_mode, load_meta = build_or_load_overlap_activity(OVERLAP_ACTIVITY_PATH, force_rebuild=FORCE_REBUILD)
activity_df["activity_date"] = pd.to_datetime(activity_df.get("activity_date"), errors="coerce")
activity_df["vend_amount_rs"] = pd.to_numeric(activity_df.get("vend_amount_rs"), errors="coerce")
activity_df["kwh"] = pd.to_numeric(activity_df.get("kwh"), errors="coerce")
activity_df["kvah"] = pd.to_numeric(activity_df.get("kvah"), errors="coerce")

print(f"Load mode: {load_mode}")
print(load_meta.get("note", ""))
display(activity_df.head())
print("Available columns:")
print(activity_df.columns.tolist())

## Basic EDA

The cohort below uses whichever key was selected from the four candidate options.

In [ ]:
cohort_entities = activity_df[available_columns(activity_df, PROFILE_COLUMNS)].drop_duplicates(subset=["selected_key_value"]).copy()
vend_rows = activity_df[activity_df["activity_source"] == "vend"].copy()
consumption_rows = activity_df[activity_df["activity_source"] == "consumption"].copy()

summary_metrics = pd.DataFrame([
    {
        "selected_key": SELECTED_KEY_COLUMN,
        "matched_keys": int(cohort_entities["selected_key_value"].nunique()) if "selected_key_value" in cohort_entities.columns else 0,
        "profiled_master_rows": int(cohort_entities.get("consumer_master_id", pd.Series(dtype="Int64")).nunique(dropna=True)),
        "vend_rows": int(len(vend_rows)),
        "consumption_rows": int(len(consumption_rows)),
        "total_vend_amount_rs": vend_rows["vend_amount_rs"].sum(),
        "total_kwh": consumption_rows["kwh"].sum(),
        "avg_vend_amount_per_txn": vend_rows["vend_amount_rs"].mean(),
        "median_vend_amount_per_txn": vend_rows["vend_amount_rs"].median(),
        "avg_daily_kwh_row": consumption_rows["kwh"].mean(),
        "median_daily_kwh_row": consumption_rows["kwh"].median(),
        "vend_date_min": vend_rows["activity_date"].min(),
        "vend_date_max": vend_rows["activity_date"].max(),
        "consumption_date_min": consumption_rows["activity_date"].min(),
        "consumption_date_max": consumption_rows["activity_date"].max(),
    }
])

display(summary_metrics)
display(vend_rows[["vend_amount_rs"]].describe().T)
display(consumption_rows[["kwh", "kvah"]].describe().T)
display(cohort_entities.head(20))

In [ ]:
entity_eda = (
    activity_df.groupby("selected_key_value", dropna=True)
    .agg(
        selected_key_column=("selected_key_column", "first"),
        master_consumername=("master_consumername", "first"),
        master_consumernumber=("master_consumernumber", "first"),
        master_meterno=("master_meterno", "first"),
        master_tariffcode=("master_tariffcode", "first"),
        master_area_type=("master_area_type", "first"),
        master_meterbalance=("master_meterbalance", "first"),
        master_profile_status=("master_profile_status", "first"),
        vend_row_count=("vend_amount_rs", lambda s: int(s.notna().sum())),
        total_vend_amount_rs=("vend_amount_rs", "sum"),
        consumption_day_count=("kwh", lambda s: int(s.notna().sum())),
        total_kwh=("kwh", "sum"),
        average_daily_kwh=("kwh", "mean"),
    )
    .reset_index()
    .sort_values(["total_vend_amount_rs", "total_kwh"], ascending=False)
)

display(entity_eda.head(20))

if "master_tariffcode" in cohort_entities.columns:
    display(cohort_entities["master_tariffcode"].value_counts(dropna=False).head(10).rename_axis("master_tariffcode").reset_index(name="entity_count"))
if "master_area_type" in cohort_entities.columns:
    display(cohort_entities["master_area_type"].value_counts(dropna=False).head(10).rename_axis("master_area_type").reset_index(name="entity_count"))
if "master_profile_status" in cohort_entities.columns:
    display(cohort_entities["master_profile_status"].value_counts(dropna=False).rename_axis("master_profile_status").reset_index(name="entity_count"))

## Daily charts

These three charts are built from the selected overlap cohort. The rupees-per-kWh ratio is calculated as `total daily vend amount / total daily kWh` on dates where total kWh is positive.

In [ ]:
daily_consumption = (
    consumption_rows.dropna(subset=["activity_date"])
    .groupby("activity_date", dropna=True)["kwh"]
    .sum()
    .reset_index(name="total_daily_kwh")
    .sort_values("activity_date")
)

daily_vend = (
    vend_rows.dropna(subset=["activity_date"])
    .groupby("activity_date", dropna=True)["vend_amount_rs"]
    .sum()
    .reset_index(name="total_daily_vend_amount")
    .sort_values("activity_date")
)

daily_combined = daily_consumption.merge(
    daily_vend,
    on="activity_date",
    how="outer",
).sort_values("activity_date")

daily_combined["total_daily_kwh"] = daily_combined["total_daily_kwh"].fillna(0)
daily_combined["total_daily_vend_amount"] = daily_combined["total_daily_vend_amount"].fillna(0)
daily_combined["avg_rupees_per_kwh"] = (
    daily_combined["total_daily_vend_amount"] / daily_combined["total_daily_kwh"].where(daily_combined["total_daily_kwh"] > 0)
)

display(daily_combined.head(20))

In [ ]:
fig_consumption = px.line(
    daily_consumption,
    x="activity_date",
    y="total_daily_kwh",
    markers=True,
    title="Total daily consumption for overlap cohort",
)
fig_consumption.update_layout(template="plotly_white", xaxis_title="Date", yaxis_title="Total daily kWh")
fig_consumption.show()

In [ ]:
fig_vend = px.line(
    daily_vend,
    x="activity_date",
    y="total_daily_vend_amount",
    markers=True,
    title="Total daily vend amount for overlap cohort",
)
fig_vend.update_layout(template="plotly_white", xaxis_title="Date", yaxis_title="Total daily vend amount (Rs)")
fig_vend.show()

In [ ]:
rupees_per_kwh_daily = daily_combined.dropna(subset=["avg_rupees_per_kwh"]).copy()

fig_ratio = px.line(
    rupees_per_kwh_daily,
    x="activity_date",
    y="avg_rupees_per_kwh",
    markers=True,
    title="Average rupees per kWh received per day",
)
fig_ratio.update_layout(template="plotly_white", xaxis_title="Date", yaxis_title="Rs per kWh")
fig_ratio.show()

## Optional refresh

If the processed source files are refreshed later, set `FORCE_REBUILD = True` and rerun the notebook to regenerate the saved overlap parquet for the selected key.